In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
import torch

dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [2]:
# q1
def combine(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

dataset = dataset.map(combine)

print(len(dataset[51]["combined_text"]))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


In [3]:
# q2
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522


In [4]:
# q3
print(tokenizer.sep_token_id)

102


In [5]:
# q4
prompts = list(dataset["prompt"])

tokens = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(tokens["input_ids"].shape)

torch.Size([2000, 128])


In [6]:
# q5
hidden_size = 768
num_heads = 12

head_dim = hidden_size // num_heads

print(head_dim)

64


In [7]:
# q6
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained("bert-base-uncased")

text = dataset[0]["prompt"]

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])


In [8]:
# q7
cls_embedding = outputs.last_hidden_state[0, 0]

answer = cls_embedding[:5].sum().item()

print(round(answer, 4))

-1.2001


In [9]:
# q8
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

sentence = "Light-ion fusion is a technique."

inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

fusion_index = tokens.index("fusion")

attention = outputs.attentions[-1][0, 0]

print(round(attention[0, fusion_index].item(), 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
0.1025


In [10]:
# q9
from sentence_transformers import SentenceTransformer, util

# Load MiniLM model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt = dataset[0]["prompt"]
option_b = dataset[0]["B"]

prompt_embedding = model.encode(prompt, convert_to_tensor=True)
option_embedding = model.encode(option_b, convert_to_tensor=True)

similarity = util.cos_sim(prompt_embedding, option_embedding)

print(round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658


In [11]:
# q10
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch

In [12]:
minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def map_at_3(preds, ans):

    if ans in preds:
        return 1 / (preds.index(ans) + 1)

    return 0

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
def tfidf_top3(prompt, options):

    docs = [prompt] + options

    vectorizer = TfidfVectorizer()

    X = vectorizer.fit_transform(docs)

    sim = cosine_similarity(X[0:1], X[1:]).flatten()

    return np.argsort(sim)[::-1][:3]

In [14]:
letters = ["A","B","C","D","E"]

map_score = 0
improved_count = 0

for row in dataset:

    prompt = row["prompt"]

    options = [row["A"], row["B"], row["C"], row["D"], row["E"]]

    answer = row["answer"]

    prompt_emb = minilm_model.encode(prompt, convert_to_tensor=True)

    option_embs = minilm_model.encode(options, convert_to_tensor=True)

    sims = util.cos_sim(prompt_emb, option_embs)[0]

    mini_pred = torch.argsort(sims, descending=True)[:3].tolist()
    mini_pred = [letters[i] for i in mini_pred]

    tfidf_pred = tfidf_top3(prompt, options)
    tfidf_pred = [letters[i] for i in tfidf_pred]

    map_score += map_at_3(mini_pred, answer)

    # Improved Count
    if answer not in tfidf_pred and answer in mini_pred:
        improved_count += 1

In [15]:
map_score /= len(dataset)

print("MAP@3 =", round(map_score,4))
print("Improved Count =", improved_count)

MAP@3 = 0.4231
Improved Count = 564


In [16]:
# q11
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = dataset[1]

result = classifier(
    row["prompt"],
    candidate_labels=[
        row["A"],
        row["B"],
        row["C"]
    ]
)

print(result)

# Top probability
print(round(result["scores"][0], 4))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

In [17]:
# q12
from transformers import pipeline

# Load model
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row = dataset[1]

labels = [row["A"], row["B"], row["C"]]

result_softmax = classifier(
    row["prompt"],
    candidate_labels=labels
)

# Independent Sigmoid
result_multilabel = classifier(
    row["prompt"],
    candidate_labels=labels,
    multi_label=True
)

# Sum of probabilities
softmax_sum = sum(result_softmax["scores"])
multilabel_sum = sum(result_multilabel["scores"])

# Absolute difference
difference = abs(softmax_sum - multilabel_sum)

print("Softmax Sum:", softmax_sum)
print("Multi-label Sum:", multilabel_sum)
print("Difference:", round(difference, 4))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Softmax Sum: 0.9999999701976776
Multi-label Sum: 0.0005095975611766335
Difference: 0.9995


In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# Row 0
row = dataset[0]

prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt")

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

# Decode
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Model Output:", answer)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model Output: B
